In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
# Create SparkSession
spark =  SparkSession.builder \
                    .master("spark://spark-master:7077") \
                    .appName("example") \
                    .config("spark.executor.memory", "2g") \
                    .getOrCreate()
# spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 10:18:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext

In [3]:
import pyspark.sql.functions as sf
from pyspark.sql import Row
df = spark.createDataFrame([Row(name="Alice", age=2), Row(name="Bob", age=5)])
df2 = spark.createDataFrame([Row(name="Tom", height=80), Row(name="Bob", height=85)])
df3 = spark.createDataFrame([
    Row(name="Alice", age=10, height=80),
    Row(name="Bob", age=5, height=None),
    Row(name="Tom", age=None, height=None),
    Row(name=None, age=None, height=None),
])

In [5]:
from pyspark.sql import Row
df = spark.createDataFrame([Row(name="Alice", age=2), Row(name="Bob", age=5)])

In [4]:
# only narrow transformation
df.show()

+-----+---+
| name|age|
+-----+---+
|Alice|  2|
|  Bob|  5|
+-----+---+



In [5]:
# wide transformation
df.count()

2

In [7]:
df.cache()

DataFrame[name: string, age: bigint]

In [9]:
df.persist()

26/02/26 06:31:10 WARN CacheManager: Asked to cache already cached data.


DataFrame[name: string, age: bigint]

In [15]:
from pyspark import StorageLevel
df3.persist(StorageLevel.MEMORY_AND_DISK)

DataFrame[name: string, age: bigint, height: bigint]

In [16]:
df3.show()

+-----+----+------+
| name| age|height|
+-----+----+------+
|Alice|  10|    80|
|  Bob|   5|  NULL|
|  Tom|NULL|  NULL|
| NULL|NULL|  NULL|
+-----+----+------+



In [4]:
parquet_df=spark.read.parquet("/app/data/ecommerce.parquet")

In [10]:
parquet_df.show(4)

+--------+-------+----------+--------+------+--------+--------------+------------+-------------------+-------+
|order_id|user_id|product_id|category| price|quantity|payment_method|order_status|           order_ts|country|
+--------+-------+----------+--------+------+--------+--------------+------------+-------------------+-------+
|       0| 492697|     20113|  sports| 493.4|       2|    debit_card|    returned|2024-06-28 00:46:30|     FR|
|       1| 499190|     95367|   books|166.82|       2|   credit_card|    returned|2021-08-28 00:19:18|     CA|
|       2| 925876|      2798|  sports|271.33|       3|   credit_card|    returned|2023-06-17 04:18:19|     UK|
|       3| 985770|     68273| fashion| 83.88|       4|           upi|      placed|2024-07-01 01:42:03|     CA|
+--------+-------+----------+--------+------+--------+--------------+------------+-------------------+-------+
only showing top 4 rows


In [8]:
from pyspark.sql.functions import col
parquet_df_filter=parquet_df.filter(col("category") == "sports")  # Transformation

In [12]:
parquet_df_filter.collect()   # Action

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 9) / 9]
Traceback (most recent call last):
  File "/opt/bitnami/spark/python/lib/py4j.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/lib/py4j.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/python/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
26/03/18 05:53:36 WARN BlockManager: Failed to fetch remote block taskresult_39 from [BlockManagerId(0, 172.18.0.5, 39829, None)] after 1 fetch failures. Most recent failure cause:
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala

In [10]:
paquet_df.cache()

DataFrame[order_id: bigint, user_id: bigint, product_id: bigint, category: string, price: double, quantity: bigint, payment_method: string, order_status: string, order_ts: timestamp_ntz, country: string]

In [9]:
paquet_df.count()

50000000

In [10]:
paquet_dfff.groupBy("category").count().show()

+-----------+--------+
|   category|   count|
+-----------+--------+
|      books| 9997463|
|electronics|10002586|
|     sports|10001692|
|       home|10000660|
|    fashion| 9997599|
+-----------+--------+



In [14]:
paquet_df.rdd.getNumPartitions()

9

In [4]:
paquet_df.distinct().count()

50000000

In [6]:
paquet_df=paquet_df.repartition(2)

In [9]:
paquet_df = paquet_df.repartition(2).cache()

In [ ]:
from pyspark import StorageLevel
paquet_df.persist(StorageLevel.MEMORY_ONLY)

In [ ]:
# Transformations
sports_orders = paquet_df.filter("category = 'sports'")
high_value_orders = sports_orders.filter("price > 200")

In [ ]:
# Action 1: show
high_value_orders.show()  
# This triggers computation and persists high_value_orders if its parent was persisted

# Action 2: count
count = high_value_orders.count()  

# Action 3: collect
rows = high_value_orders.collect()

In [ ]:
from pyspark.sql.functions import count, sum, avg

df_sum = paquet_df.select(sum("price")).collect()[0][0] # Returns the sum of 'Salary' column
print(df_sum)

### skewd Data

In [3]:
csv_df=spark.read.csv("/app/data/skewed_dataset.csv")

In [5]:
skewed_df = csv_df.partitionBy("_c1")  # one partition per key

In [5]:
skewed_df = csv_df.repartition("_c1")  # one partition per key

In [6]:
skewed_df.groupBy("_c1").count().show()

+-------+-------+
|    _c1|  count|
+-------+-------+
|     MX| 133494|
|     CN| 133829|
|     CA| 133255|
|     GB| 132891|
|     BR| 133426|
|     DE| 132870|
|     US|4800112|
|     IN| 133535|
|     FR| 133302|
|     JP| 133286|
|country|      1|
+-------+-------+



#### From Table

In [5]:
paquet_df.createOrReplaceTempView("ttable")

In [1]:
df=spark.sql("SELECT order_id, user_id, product_id, category, price FROM (SELECT * FROM ttable WHERE order_status='returned') WHERE price>100")
df.explain("codegen")

In [2]:
df.explain("formatted")

In [3]:
df.explain("cost")

#### For Spark-history-server

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeepSparkDebug") \
    .master("spark://spark-master:7077") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "/tmp/spark-events") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 09:27:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.conf.set("spark.sql.adaptive.enabled", "False")
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")

### Joins

In [ ]:
from pyspark.sql.functions import broadcast

In [ ]:
import pyspark.sql.functions as sf
from pyspark.sql import Row
df = spark.createDataFrame([Row(name="Alice", age=2), Row(name="Bob", age=5)])
df2 = spark.createDataFrame([Row(name="Tom", height=80), Row(name="Bob", height=85)])
df3 = spark.createDataFrame([
    Row(name="Alice", age=10, height=80),
    Row(name="Bob", age=5, height=None),
    Row(name="Tom", age=None, height=None),
    Row(name=None, age=None, height=None),
])

In [ ]:
joined = df.join(broadcast(df2), "name")
joined.show() 

In [ ]:
joined = paquet_df.join(product_df, paquet_df.category == product_df.category, "left")

In [ ]:
joined = paquet_df.join(df, df.age == paquet_df.product_id, "inner")
joined.show()

# data Skewd

In [ ]:
from pyspark.sql.functions import col, rand

# Create a skewed dataset
data = []

# Key "A" will dominate (skewed), other keys less frequent
for i in range(10000):
    key = "A" if i < 9000 else "B" if i < 9500 else "C"
    value = i
    data.append((key, value))

df = spark.createDataFrame(data, ["key", "value"])

# Show basic stats
df.groupBy("key").count().show()

# Optional: randomize values to make UI observation easier
df = df.withColumn("random_val", rand())

# Trigger an action to see tasks in Spark UI
df.groupBy("key").sum("value").collect()

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", True)

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 3)

df.groupBy("key").sum("value").collect()

### Spark Word Count Example

In [8]:
sc = spark.sparkContext

rdd = sc.textFile("/app/data/test.txt")
print("initial partition count:" + str(rdd.getNumPartitions()))

initial partition count:2


In [9]:
repar_rdd = rdd.repartition(4)
print("re-partition count:" + str(repar_rdd.getNumPartitions()))


re-partition count:4


In [10]:
# rdd.coalesce(3)
for line in rdd.collect():
    print(line)

Apache Spark is a fast and general purpose cluster computing system
Spark provides high level APIs in Java Scala Python and R
Spark also supports a rich set of higher level tools
Spark SQL is used for structured data processing
Spark Streaming enables scalable and fault tolerant stream processing
Apache Hadoop is a framework for distributed storage and processing
MapReduce is the original processing model for Hadoop
Spark is faster than Hadoop MapReduce because of in memory processing
a distributed system requires fault tolerance and scalability
a good data engineer understands both batch and stream processing


In [28]:
# rdd flatMap transformation
rdd2 = rdd.flatMap(lambda f: f.split(" "))
rdd2.collect()[:10]

['Apache',
 'Spark',
 'is',
 'a',
 'fast',
 'and',
 'general',
 'purpose',
 'cluster',
 'computing']

In [29]:
# Create a Tuple by adding 1 to each word
rdd3 = rdd2.map(lambda m: (m, 1))
rdd3.collect()[:8]

[('Apache', 1),
 ('Spark', 1),
 ('is', 1),
 ('a', 1),
 ('fast', 1),
 ('and', 1),
 ('general', 1),
 ('purpose', 1)]

In [30]:
# Filter transformation
rdd4 = rdd3.filter(lambda a: a[0].startswith("a"))
rdd4.collect()[:8]

[('a', 1),
 ('and', 1),
 ('and', 1),
 ('also', 1),
 ('a', 1),
 ('and', 1),
 ('a', 1),
 ('and', 1)]

In [31]:
# ReduceBy transformation
rdd5 = rdd3.reduceByKey(lambda a, b: a + b)
rdd5.collect()[:8]

[('fast', 1),
 ('and', 6),
 ('general', 1),
 ('computing', 1),
 ('high', 1),
 ('level', 2),
 ('Java', 1),
 ('Scala', 1)]

In [15]:
# Swap word,count and sortByKey transformation
rdd6 = rdd5.map(lambda a: (a[1], a[0])).sortByKey()
print("Final Result")

Final Result


In [32]:
# Action - foreach
rdd6.collect()[:8]

[(1, 'model'),
 (1, 'faster'),
 (1, 'than'),
 (1, 'because'),
 (1, 'memory'),
 (1, 'requires'),
 (1, 'tolerance'),
 (1, 'good')]

In [17]:
# Action - count
print("Count : " + str(rdd6.count()))

Count : 59


In [18]:
# Action - first
first_rec = rdd6.first()
print("First Record : " + str(first_rec[0]) + "," + first_rec[1])

First Record : 1,purpose


In [19]:
# Action - max
dat_max = rdd6.max()
print("Max Record : " + str(dat_max[0]) + "," + dat_max[1])

Max Record : 6,processing


In [20]:
# Action - reduce
total_word_count = rdd6.reduce(lambda a, b: (a[0] + b[0], a[1]))
print("dataReduce Record : " + str(total_word_count[0]))

dataReduce Record : 96


In [21]:
# Action - take
data3 = rdd6.take(3)
for f in data3:
    print("data3 Key:" + str(f[0]) + ", Value:" + f[1])

data3 Key:1, Value:purpose
data3 Key:1, Value:cluster
data3 Key:1, Value:provides


In [22]:
# Action - collect
data = rdd6.collect()
for f in data:
    print("Key:" + str(f[0]) + ", Value:" + f[1])

Key:1, Value:model
Key:1, Value:faster
Key:1, Value:than
Key:1, Value:because
Key:1, Value:memory
Key:1, Value:requires
Key:1, Value:tolerance
Key:1, Value:good
Key:1, Value:understands
Key:1, Value:batch
Key:1, Value:fast
Key:1, Value:general
Key:1, Value:computing
Key:1, Value:high
Key:1, Value:Java
Key:1, Value:Scala
Key:1, Value:Python
Key:1, Value:R
Key:1, Value:supports
Key:1, Value:set
Key:1, Value:SQL
Key:1, Value:used
Key:1, Value:structured
Key:1, Value:enables
Key:1, Value:tolerant
Key:1, Value:framework
Key:1, Value:storage
Key:1, Value:purpose
Key:1, Value:cluster
Key:1, Value:provides
Key:1, Value:APIs
Key:1, Value:also
Key:1, Value:rich
Key:1, Value:higher
Key:1, Value:tools
Key:1, Value:Streaming
Key:1, Value:scalable
Key:1, Value:the
Key:1, Value:original
Key:1, Value:scalability
Key:1, Value:engineer
Key:1, Value:both
Key:2, Value:Apache
Key:2, Value:system
Key:2, Value:in
Key:2, Value:data
Key:2, Value:stream
Key:2, Value:MapReduce
Key:2, Value:of
Key:2, Value:distri